In [1]:
# ============================================================
# RF_PI (Physics-Informed Random Forest)
# Performance evaluation across 50 station-level splits
# Station-Level 70:30 Training-Test Split Across 50 Random Seeds
#
# Author: Junyoung Lee
# Affiliation: Ulsan National Institute of Science and Technology (UNIST)
# Email: junyounglee@unist.ac.kr
# ============================================================

import numpy as np
import pandas as pd
import os
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ------------------------------------------------------------
# Repository paths
# ------------------------------------------------------------
# This notebook can be run from either the repository root
# or the notebooks directory.
current_dir = Path.cwd()
project_dir = (
    current_dir.parent
    if current_dir.name == 'notebooks'
    else current_dir
)

data_dir = project_dir / 'data'
output_dir = project_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

input_file = data_dir / 'Total data_for submission.csv'
total_data = pd.read_csv(input_file)

# Use the absolute value of U.ratio
total_data['U.ratio'] = np.abs(total_data['U.ratio'])

station_col = 'SSN'
seeds = list(range(1, 51))

feature_sets = {
    'Set1': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle'],
    'Set2': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'S.Lat', 'S.Long'],
    'Set3': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'slope_500m'],
    'Set4': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'S.Lat', 'S.Long', 'slope_500m']
}

# -----------------------------
# Output folders
# -----------------------------
prediction_dir = os.path.join(output_dir, 'seed_predictions_RF_PI')
os.makedirs(prediction_dir, exist_ok=True)

# Directory for feature-importance results from individual seeds
importance_dir = os.path.join(output_dir, 'seed_feature_importance_RF_PI')
os.makedirs(importance_dir, exist_ok=True)

# -----------------------------
# Prepare PI target
# -----------------------------
total_data = total_data[
    (total_data['Vs30_mea'] > 0) &
    (total_data['Vs30_f0'] > 0)
].copy()

total_data['log_res'] = (
    np.log(total_data['Vs30_mea']) -
    np.log(total_data['Vs30_f0'])
)

all_results = []
all_importances = []

# -----------------------------
# Repeat station-level split
# -----------------------------
for seed in seeds:

    np.random.seed(seed)

    stations = np.array(
        sorted(total_data[station_col].unique())
    )
    np.random.shuffle(stations)

    n_train = int(len(stations) * 0.7)

    train_stations = stations[:n_train]
    test_stations = stations[n_train:]

    data1 = total_data[
        total_data[station_col].isin(train_stations)
    ].copy()

    data2 = total_data[
        total_data[station_col].isin(test_stations)
    ].copy()

    y_test = data2['Vs30_mea'].values
    y_pred_p = data2['Vs30_f0'].values

    # -----------------------------
    # P-wave metrics
    # -----------------------------
    rmse_p = np.sqrt(
        mean_squared_error(y_test, y_pred_p)
    )
    r2_p = r2_score(y_test, y_pred_p)
    mae_p = mean_absolute_error(y_test, y_pred_p)
    bias_p = np.mean(y_pred_p - y_test)

    all_results.append({
        'seed': seed,
        'model': 'P-wave',
        'feature_set': 'P-wave',
        'RMSE': rmse_p,
        'R2': r2_p,
        'MAE': mae_p,
        'Bias': bias_p,
        'Delta_RMSE': 0.0,
        'Delta_R2': 0.0,
        'Delta_MAE': 0.0
    })

    pred_df = data2.copy()
    pred_df['P-wave_pred'] = y_pred_p

    # Feature-importance results for the current seed
    seed_importances = []

    # -----------------------------
    # RF_PI for Set1-Set4
    # -----------------------------
    for set_name, features in feature_sets.items():

        train_sub = data1[
            features + ['log_res']
        ].dropna()

        test_sub = data2[
            features
        ].dropna()

        idx = test_sub.index

        X_train = train_sub[features]
        y_train = train_sub['log_res']
        X_test = test_sub[features]

        model = RandomForestRegressor(
            n_estimators=500,
            random_state=seed,
            n_jobs=-1,
            bootstrap=True,
            min_samples_split=5,
            min_samples_leaf=4,
            max_features=3,
            max_depth=7,
            oob_score=True
        )

        model.fit(X_train, y_train)

        # -----------------------------
        # Feature importance
        # -----------------------------
        for feature, importance in zip(
            features,
            model.feature_importances_
        ):

            importance_result = {
                'seed': seed,
                'model': 'RF_PI',
                'feature_set': set_name,
                'feature': feature,
                'importance': importance
            }

            all_importances.append(importance_result)
            seed_importances.append(importance_result)

        log_res_pred = model.predict(X_test)

        # Physics-informed prediction
        y_pred = (
            data2.loc[idx, 'Vs30_f0'].values *
            np.exp(log_res_pred)
        )

        y_true = data2.loc[idx, 'Vs30_mea'].values
        y_p_sub = data2.loc[idx, 'Vs30_f0'].values

        # RF_PI metrics
        rmse = np.sqrt(
            mean_squared_error(y_true, y_pred)
        )
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        bias = np.mean(y_pred - y_true)

        # P-wave metrics on the same subset
        rmse_p_sub = np.sqrt(
            mean_squared_error(y_true, y_p_sub)
        )
        r2_p_sub = r2_score(y_true, y_p_sub)
        mae_p_sub = mean_absolute_error(
            y_true,
            y_p_sub
        )

        all_results.append({
            'seed': seed,
            'model': 'RF_PI',
            'feature_set': set_name,
            'RMSE': rmse,
            'R2': r2,
            'MAE': mae,
            'Bias': bias,
            'Delta_RMSE': rmse - rmse_p_sub,
            'Delta_R2': r2 - r2_p_sub,
            'Delta_MAE': mae - mae_p_sub
        })

        pred_df.loc[
            idx,
            f'{set_name}_pred'
        ] = y_pred

    # -----------------------------
    # Save prediction by seed
    # -----------------------------
    pred_df.to_csv(
        os.path.join(
            prediction_dir,
            f'seed_{seed:02d}_predictions.csv'
        ),
        index=False
    )

    # -----------------------------
    # Save importance by seed
    # -----------------------------
    seed_importance_df = pd.DataFrame(
        seed_importances
    )

    seed_importance_df.to_csv(
        os.path.join(
            importance_dir,
            f'seed_{seed:02d}_feature_importance.csv'
        ),
        index=False
    )

# -----------------------------
# Save model results
# -----------------------------
results_df = pd.DataFrame(all_results)

results_df = results_df[
    [
        'seed',
        'model',
        'feature_set',
        'RMSE',
        'R2',
        'MAE',
        'Bias',
        'Delta_RMSE',
        'Delta_R2',
        'Delta_MAE'
    ]
]

results_df.to_csv(
    os.path.join(
        output_dir,
        '02-RF_PI_results.csv'
    ),
    index=False
)



In [2]:
# -----------------------------
# Summary of mean performance across seeds
# -----------------------------
summary_df = results_df.groupby(['model', 'feature_set']).agg({
    'RMSE': 'mean',
    'R2': 'mean',
    'MAE': 'mean',
    'Bias': 'mean',
    'Delta_RMSE': 'mean',
    'Delta_R2': 'mean',
    'Delta_MAE': 'mean'
}).reset_index()

summary_df.to_csv(
    os.path.join(output_dir, '02-RF_PI_summary_mean.csv'),
    index=False
)

print(summary_df)

    model feature_set        RMSE        R2         MAE       Bias  \
0  P-wave      P-wave  238.051324  0.154893  183.892736  19.050835   
1   RF_PI        Set1  237.272226  0.184985  181.075336 -10.437494   
2   RF_PI        Set2  212.509110  0.351912  161.522769 -20.383456   
3   RF_PI        Set3  222.741555  0.275846  168.419722 -14.714037   
4   RF_PI        Set4  211.137446  0.358134  158.882029 -21.476143   

   Delta_RMSE  Delta_R2  Delta_MAE  
0    0.000000  0.000000   0.000000  
1   -0.779098  0.030092  -2.817400  
2  -25.542214  0.197019 -22.369967  
3  -15.309769  0.120953 -15.473013  
4  -26.913878  0.203241 -25.010707  


In [3]:
# -----------------------------
# Save all feature importances
# -----------------------------
importance_df = pd.DataFrame(all_importances)

importance_df = importance_df[
    [
        'seed',
        'model',
        'feature_set',
        'feature',
        'importance'
    ]
]

importance_df.to_csv(
    os.path.join(
        output_dir,
        '03-RF_PI_feature_importance_by_seed.csv'
    ),
    index=False
)

# -----------------------------
# Save summary statistics across seeds
# -----------------------------
importance_summary_df = (
    importance_df
    .groupby(
        ['model', 'feature_set', 'feature'],
        as_index=False
    )
    .agg(
        importance_mean=('importance', 'mean'),
        importance_sd=('importance', 'std'),
        importance_median=('importance', 'median'),
        importance_min=('importance', 'min'),
        importance_max=('importance', 'max'),
        n_seed=('seed', 'nunique')
    )
)

importance_summary_df.to_csv(
    os.path.join(
        output_dir,
        '03-RF_PI_feature_importance_summary.csv'
    ),
    index=False
)

print(results_df.head(15))
print(importance_df.head(15))
print(importance_summary_df)

    seed   model feature_set        RMSE        R2         MAE        Bias  \
0      1  P-wave      P-wave  260.316949 -0.823874  202.625316  109.491139   
1      1   RF_PI        Set1  232.386820 -0.453493  182.436979  102.427154   
2      1   RF_PI        Set2  203.796423 -0.117849  159.140294   59.199182   
3      1   RF_PI        Set3  208.916347 -0.174721  166.865680   88.592796   
4      1   RF_PI        Set4  186.494769  0.063898  149.934370   67.886967   
5      2  P-wave      P-wave  197.224723  0.524975  142.609756  -11.609756   
6      2   RF_PI        Set1  199.334155  0.514759  145.433840  -24.858333   
7      2   RF_PI        Set2  167.440485  0.657615  125.645381   19.772826   
8      2   RF_PI        Set3  197.744338  0.522469  139.971015  -41.016521   
9      2   RF_PI        Set4  153.130857  0.713635  105.465577  -21.398113   
10     3  P-wave      P-wave  252.807570  0.308920  192.029674  -89.103858   
11     3   RF_PI        Set1  275.079639  0.181789  210.827517 -